---
# Curso: Data Mining - Semana 7
Docente: Soledad Espezúa (s.espezual@up.edu.pe)

---


# <font color=green>Laboratorio de repaso integrador</font>
## <font color=blue>Caso: clientes de una empresa de telecomunicaciones</font>

En este laboratorio usaremos un caso de telecomunicaciones para integrar las decisiones
trabajadas hasta ahora: comprensión del problema, integración, calidad, faltantes, EDA,
valores atípicos, codificación, variables derivadas, discretización, escalamiento y
construcción de una matriz analítica.

La idea no es seguir una lista de técnicas. Primero observamos los datos, entendemos qué
significan y recién después decidimos qué transformar.

<font color=red>**Recuerda:** una transformación debe responder al significado de la variable
y al objetivo de la unidad de análisis, no a una rutina automática.</font>


## <font color=green>1. Entendimiento del negocio</font>

En telecomunicaciones, cuando el mercado está bastante cubierto, captar nuevos clientes puede ser
costoso. Por ello, una preocupación importante de las empresas es la **retención de clientes**.

Uno de los indicadores utilizados es el **churn** o abandono: clientes que dejan de mantener
activa su línea.

En este caso queremos preparar una base que permita explorar características demográficas,
de consumo y de deuda que podrían estar relacionadas con el abandono.

Para simplificar el caso asumiremos que:

> **cada cliente tiene una sola línea**

Por tanto, nuestra unidad de análisis será:

> **una fila = un cliente**


## <font color=green>2. Entendimiento de los datos</font>

Disponemos de dos fuentes que describen a los mismos clientes:

1. `Maestro_Clientes.csv`: información demográfica.
2. `Datos_Clientes.csv`: información de consumo, deuda y estado de la línea.

Ambas tablas se relacionan mediante `CLIENT_ID`.


### <font color=blue>Diccionario de datos: Maestro_Clientes.csv</font>

- `CLIENT_ID`: identificador único del cliente.
- `GENERO`: género del cliente (`M` masculino, `F` femenino).
- `EDAD`: edad del cliente en años cumplidos.
- `NSE`: nivel socioeconómico del cliente.
- `ESTADOCIVIL`: estado civil (`S` soltero, `C` casado, `V` viudo).
- `DEPARTAMENTO`: lugar de residencia declarado por el cliente.
- `CAPDEUDA`: indicador de capacidad de endeudamiento del cliente.


### <font color=blue>Diccionario de datos: Datos_Clientes.csv</font>

- `CLIENT_ID`: identificador único del cliente.
- `RENTA`: renta básica del plan.
- `USO_SMS`: número de SMS enviados en el mes.
- `DEUDA_ANT`: deuda facturada en el mes anterior.
- `MIN_CALLCENTER`: minutos de llamadas al centro de atención.
- `ANTIGUEDAD`: antigüedad del cliente en meses.
- `USO_MB`: total de MB consumidos en el periodo.
- `DEUDA_ACT`: deuda facturada en el mes.
- `USO_APP`: MB consumidos en la aplicación de la operadora.
- `MIN_ENT_3M`: llamadas recibidas por el cliente en los últimos 3 meses.
- `MIN_SAL_3M`: llamadas realizadas por el cliente en los últimos 3 meses.
- `CHURN`: `0` línea inactiva, `1` línea suspendida y `2` línea activa al cierre.


## <font color=green>3. Integración de las fuentes</font>

Como las dos tablas tienen la misma unidad de análisis y se relacionan por `CLIENT_ID`,
podemos integrarlas directamente.

Usaremos `validate="one_to_one"` porque esperamos que cada cliente aparezca una sola vez
en cada fuente.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.precision", 2)
sns.set_theme(style="ticks")

maestro = pd.read_csv("Maestro_Clientes.csv", sep=";")
datos = pd.read_csv("Datos_Clientes.csv", sep=";")

print("Maestro:", maestro.shape)
print("Datos:", datos.shape)

display(maestro.head())
display(datos.head())


In [ ]:
# Integramos las dos fuentes usando CLIENT_ID.
df = maestro.merge(
    datos,
    on="CLIENT_ID",
    how="inner",
    validate="one_to_one"
)

print("Dimensiones del tablón integrado:", df.shape)
display(df.head())


### <font color=blue>Limpieza estructural</font>

Antes de analizar los contenidos, revisamos problemas estructurales evidentes:
filas completamente vacías, columnas completamente vacías y duplicados exactos.

En este conjunto la función no debería cambiar las dimensiones, pero deja explícita una
comprobación que puede ser útil en otros proyectos.


In [ ]:
def limpieza_estructural(df):

    df = df.dropna(how="all")           # Elimina filas completamente vacías.
    df = df.dropna(axis=1, how="all")   # Elimina columnas completamente vacías.
    df = df.drop_duplicates()           # Conserva una sola copia de duplicados exactos.

    return df.copy()


print("Antes:", df.shape)
df = limpieza_estructural(df)
print("Después:", df.shape)

## <font color=green>4. Definimos la variable de interés</font>

Nuestro interés es analizar **si el cliente abandonó o no el servicio**.

Primero revisamos `CHURN`, que contiene tres estados.


In [ ]:
df.CHURN.value_counts()

Como nuestro interés es identificar si el cliente **abandonó o no el servicio**, transformaremos
`CHURN` en una variable binaria.

Originalmente:

- `CHURN = 0`: línea inactiva.
- `CHURN = 1`: línea suspendida.
- `CHURN = 2`: línea activa.

Para este análisis construiremos:

- `TARGET = 1`: el cliente abandonó (`CHURN = 0`).
- `TARGET = 0`: el cliente no abandonó (`CHURN = 1` o `CHURN = 2`).

De esta manera, `TARGET` representa directamente la pregunta:

**¿El cliente abandonó o no el servicio?**


In [ ]:
# Transformamos CHURN en una variable binaria de abandono.
# 0 = línea inactiva  → TARGET = 1
# 1 = suspendida      → TARGET = 0
# 2 = activa          → TARGET = 0

df["TARGET"] = df["CHURN"].map({
    0: 1,
    1: 0,
    2: 0
})

# Revisamos la distribución de la nueva variable.
display(df["TARGET"].value_counts())
display((df["TARGET"].value_counts(normalize=True) * 100).round(1))

# Para este objetivo ya no necesitamos CHURN como característica.
df = df.drop(columns="CHURN")


En este caso, se puede observar que existe un desbalance entre las clases (66% vs 33%). Este desbalance no es muy fuerte.

## <font color=green>5. Revisamos los tipos de datos</font>

Antes de limpiar o transformar, verificamos si el tipo detectado por pandas es coherente
con el significado de cada variable según el diccionario.

Por ejemplo, `EDAD` debería ser una variable numérica.


In [ ]:
# Revisamos cómo están almacenadas las variables.
display(df.dtypes.to_frame("dtype"))

## <font color=green>6. Corregimos variables con un tipo de dato incoherente</font>

`EDAD` aparece almacenada como `object`, aunque conceptualmente debería ser numérica.

Antes de convertirla, revisamos qué valores están provocando esta situación.


In [ ]:
# Observamos los registros de EDAD que no contienen un valor numérico.
df.loc[df["EDAD"].str.isnumeric() == False]

In [ ]:
# Convertimos EDAD a numérico.
# Los valores que no puedan convertirse se transforman en faltantes (NaN).
df["EDAD"] = pd.to_numeric(
    df["EDAD"],
    errors="coerce"
)

In [ ]:
df["EDAD"].dtype #Verficamos

<font color=red>**Recuerda:** el tipo almacenado por pandas y el tipo conceptual de la variable
no son necesariamente lo mismo. `CLIENT_ID`, por ejemplo, puede estar almacenado como número
pero sigue siendo un identificador.</font>


## <font color=green>7. Exploramos las variables numéricas</font>

Una vez corregido el tipo de `EDAD`, revisamos las variables numéricas.

Los estadísticos descriptivos y los boxplots permiten obtener una primera visión de:

- rango y dispersión;
- variables constantes;
- posibles valores faltantes;
- valores extremos que merecen revisión.

Primero observamos; después decidimos qué tratar.


In [ ]:
# Identificamos las variables almacenadas como numéricas.
columnas_numericas = df.select_dtypes(include=np.number).columns.tolist()

print(columnas_numericas)

# Revisamos sus principales estadísticos descriptivos.
df[columnas_numericas].describe().T

### <font color=blue>Revisión gráfica de las variables numéricas</font>

Para los boxplots excluimos `CLIENT_ID` y `TARGET`, porque son un identificador y una variable
binaria de interés, no magnitudes numéricas que queramos analizar mediante este gráfico.

Usamos escalas independientes (`sharey=False`) porque las variables tienen rangos muy diferentes.




```
#list comprehension
columnas_boxplot = [col for col in columnas_numericas if col not in ["CLIENT_ID", "TARGET"]]
```



In [ ]:
# Excluimos el identificador y TARGET de la revisión con boxplots.
columnas_boxplot = []

for col in columnas_numericas:
    if col not in ["CLIENT_ID", "TARGET"]:
        columnas_boxplot.append(col)

#Mostramos un boxplot por cada variable
df[columnas_boxplot].plot(
    kind="box",
    subplots=True,
    layout=(3, 5),
    sharex=True,
    sharey=False,
    figsize=(10,4)
)
plt.tight_layout()
plt.show()

A partir de esta primera revisión podemos detectar situaciones que requieren análisis adicional.

Por ejemplo:

- `NSE` no presenta variación.
- `EDAD` muestra valores que deben revisarse según las reglas del negocio.
- `RENTA` presenta valores extremos que posteriormente analizaremos con reglas estadísticas.
- En algunas variables el `count` de `describe()` es menor que el número total de filas, lo que indica faltantes.

<font color=red>**Recuerda:** el boxplot ayuda a detectar situaciones que merecen revisión;
no decide automáticamente qué valores deben eliminarse o modificarse.</font>


### <font color=blue>7.1 NSE: categórica por significado, numérica por almacenamiento</font>

Según el diccionario, `NSE` representa el nivel socioeconómico, por lo que conceptualmente
es una variable categórica ordinal aunque esté almacenada mediante un código numérico.

En `describe()` y en el boxplot observamos que todos los registros de `NSE` tienen el mismo valor. Por tanto, es una variable constante.


In [ ]:
# Confirmamos que NSE tiene un solo valor en toda la base.
display(df["NSE"].value_counts())

print( "Número de valores diferentes:", df["NSE"].nunique())

# Al no aportar variabilidad, la excluimos del análisis.
df = df.drop(columns="NSE")

### <font color=blue>7.2 EDAD: valores inconsistentes y extremos</font>

El boxplot permite observar rápidamente valores de edad que merecen revisión.

Aplicaremos dos decisiones diferentes:

1. **Regla del negocio:** un cliente titular debe tener al menos 18 años. Los valores menores se llevarán al mínimo permitido de 18.
2. **Tratamiento de valores muy altos:** limitaremos a 80 las edades superiores a 80 para reducir la influencia de valores extremos.

Estas decisiones responden a razones distintas: la primera se basa en una regla del negocio; la segunda es una decisión de tratamiento para simplificar el análisis.


In [ ]:
# Revisamos los registros que quedan fuera de los límites definidos para este ejercicio.
df.loc[(df["EDAD"] < 18) | (df["EDAD"] > 80), ["CLIENT_ID", "EDAD"]].sort_values("EDAD")

In [ ]:
# Regla de negocio:
# un cliente titular debe tener al menos 18 años.
df.loc[df["EDAD"] < 18, "EDAD"] = 18

# Los valores superiores a 80 se llevan a 80.
# Esto NO significa que una persona mayor de 80 sea imposible.
df.loc[df["EDAD"] > 80, "EDAD"] = 80

# Los valores no convertibles a número quedaron como NaN.
# Los completamos con la mediana después de aplicar las reglas anteriores.
df["EDAD"] = df["EDAD"].fillna(df["EDAD"].median())

### <font color=blue>Verificamos el tratamiento de EDAD</font>


In [ ]:
plt.figure(figsize=(5, 2))

sns.boxplot(
    data=df,
    x="EDAD"
)

plt.title("EDAD después del tratamiento")
plt.show()

display(df["EDAD"].describe())

<font color=red>**Recuerda:** el límite inferior de 18 responde a una regla del negocio.
El límite superior de 80 es una decisión de tratamiento de valores altos,     adoptada para este ejercicio.</font>


### <font color=blue>7.3 Revisamos los valores faltantes en variables numéricas</font>

El `count` de `describe()` ya nos permitió notar que algunas variables tienen menos observaciones.
Ahora cuantificamos esos faltantes para decidir qué hacer en cada caso.


In [ ]:
def resumen_faltantes(df):
    resumen = pd.DataFrame({
        "n_faltantes": df.isna().sum(),
        "porcentaje": df.isna().mean() * 100
    })

    return (resumen[resumen["n_faltantes"] > 0]
        .sort_values("porcentaje", ascending=False) )


# Volvemos a identificar las variables numéricas después de retirar NSE.
columnas_numericas = (
    df
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

resumen_faltantes(df[columnas_numericas])

#### <font color=blue>USO_SMS, USO_APP y MIN_CALLCENTER</font>

Los valores vacíos en estas variables significan que el servicio no fue utilizado durante el mes,
ya que estos campos se obtuvieron de cruces con tablas transaccionales.

En este caso, **cero representa una ausencia real de uso**, por lo que podemos reemplazar los
faltantes por `0`.


In [ ]:
df.fillna(
    value={
        "USO_SMS": 0,
        "USO_APP": 0,
        "MIN_CALLCENTER": 0
    },
    inplace=True
)

#### <font color=blue>DEUDA_ANT, DEUDA_ACT y ANTIGUEDAD</font>

En estas variables un faltante no significa necesariamente cero (ausencia de deuda), por lo que no corresponde reemplazarlos por cero.

Para este ejercicio tomaremos las siguientes decisiones:

- Si falta `DEUDA_ANT`, utilizaremos la mediana de la deuda anterior de los demás clientes.
- Si falta `DEUDA_ACT`, utilizaremos como aproximación la DEUDA_ANT del mismo cliente, asumiendo que el monto facturado puede mantenerse similar entre dos meses consecutivos.




In [ ]:
# En DEUDA_ANT, un valor faltante no significa necesariamente que la deuda sea cero.
# Como no conocemos el monto del cliente, usamos una imputación simple con la mediana.
df["DEUDA_ANT"] = df["DEUDA_ANT"].fillna(df["DEUDA_ANT"].median())

# Si falta DEUDA_ACT, usamos como aproximación la deuda del mes anterior
# del mismo cliente.
df["DEUDA_ACT"] = df["DEUDA_ACT"].fillna(df["DEUDA_ANT"])

# ANTIGUEDAD: imputación simple con la mediana.
df["ANTIGUEDAD"] = df["ANTIGUEDAD"].fillna(df["ANTIGUEDAD"].median())

<font color=red>**Recuerda:** No todos los faltantes numéricos se tratan igual:
* a veces usamos 0 porque significa ausencia real de actividad;
* a veces usamos mediana;
* a veces aprovechamos una variable relacionada del mismo registro.
</font>

In [ ]:
# Verificamos si aún quedan valores faltantes
# en las variables numéricas después del tratamiento.
df[columnas_numericas].isnull().sum()

### <font color=blue>7.4 Revisamos posibles valores atípicos</font>

La revisión conjunta de los boxplots mostró que varias variables numéricas presentan valores que merecen revisión.

**Revisamos nuevamente las variables numéricas pendientes**

* CAPDEUDA, RENTA, USO_SMS, DEUDA_ANT, MIN_CALLCENTER, ANTIGUEDAD, USO_MB, DEUDA_ACT, USO_APP, MIN_ENT_3M y MIN_SAL_3M: aparecen observaciones fuera de los bigotes. Son candidatos a valores atípicos, pero podrían ser valores perfectamente válidos.



In [ ]:
columns_lst =['CAPDEUDA','RENTA', 'USO_SMS','DEUDA_ANT', 'MIN_CALLCENTER',
                'ANTIGUEDAD','USO_MB', 'DEUDA_ACT','USO_APP',
                'MIN_ENT_3M', 'MIN_SAL_3M']

#Mostramos un boxplot por cada variable
df[columns_lst].plot(
    kind="box",
    subplots=True,
    layout=(3, 5),
    sharex=True,
    sharey=False,
    figsize=(10,4)
)
plt.tight_layout()
plt.show()

Y aquí ya aparecen dos casos interesantes:

* USO_APP muestra una gran concentración en 0; eso lleva a revisar qué significan esos ceros y si conviene conservar la intensidad o crear una variable binaria como USA_APP.
* En RENTA, CAPDEUDA, DEUDA_ANT, DEUDA_ACT, USO_MB, etc. aparecen observaciones fuera de los bigotes. Esas observaciones son candidatas a revisión, pero no necesariamente errores.

In [ ]:
#Respeto a USO_APP: Primero cuantificamos cuántos ceros hay
# Revisamos qué proporción de clientes no utilizó la aplicación.
porcentaje_ceros =(df["USO_APP"] == 0).mean() * 100

print( "Porcentaje de clientes con USO_APP = 0:", round(porcentaje_ceros, 2), "%")

Como `USO_APP = 0 `significa que el cliente no utilizó la aplicación, crearemos una variable binaria que indique simplemente si la usa o no.

In [ ]:
# Creamos una variable binaria:
# 0 = no usa la aplicación
# 1 = usa la aplicación

df["USA_APP"] = (df["USO_APP"] > 0).astype(int)

df["USA_APP"].value_counts()

`USO_APP` y `USA_APP` no significan lo mismo:

- `USO_APP` conserva la intensidad de uso.
- `USA_APP` conserva solamente presencia o ausencia de uso.

En este ejercicio utilizaremos `USA_APP` en la matriz final para mostrar una posible
simplificación de la representación.

In [ ]:
# Ahora si podemos eliminar la variable original.
df = df.drop(columns = 'USO_APP')

Para revisar con mayor detalle algunos de los valores extremos, recordaremos dos reglas estadísticas de detección: IQR y z-score.

Ambas sirven para **detectar candidatos**, pero no deciden automáticamente qué hacer con ellos.
En este ejercicio utilizaremos **IQR** para tomar una decisión de tratamiento.


### <font color=blue>Regla IQR</font>


In [ ]:
def detectar_outliers_iqr(df, columna, factor=1.5):

    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)

    iqr = q3 - q1

    limite_inferior = q1 - factor * iqr
    limite_superior = q3 + factor * iqr

    mascara = (
        (df[columna] < limite_inferior) |
        (df[columna] > limite_superior)
    )

    candidatos = df.loc[mascara].copy()

    print("Variable:", columna)
    print("Límite inferior:", round(limite_inferior, 2))
    print("Límite superior:", round(limite_superior, 2))
    print("Candidatos:", candidatos.shape[0])

    return candidatos, limite_inferior, limite_superior

### <font color=blue>Regla z-score</font>


In [ ]:
def detectar_outliers_zscore(df, columna, umbral=3):

    media = df[columna].mean()
    desviacion = df[columna].std()

    # Calculamos el z-score de cada observación.
    z = (df[columna] - media) / desviacion

    # Identificamos los candidatos.
    mascara = z.abs() > umbral

    candidatos = df.loc[mascara].copy()
    candidatos["z_score"] = z[mascara]

    # Convertimos el umbral z a los valores originales de la variable.
    limite_inferior = media - umbral * desviacion
    limite_superior = media + umbral * desviacion

    print("Variable:", columna)
    print("Límite inferior:", round(limite_inferior, 2))
    print("Límite superior:", round(limite_superior, 2))
    print("Candidatos:", candidatos.shape[0])

    return candidatos, limite_inferior, limite_superior

### <font color=blue>7.5 Tomamos una decisión sobre los valores atípicos</font>

Los boxplots muestran valores extremos en varias variables. Para recordar cómo una regla estadística
puede apoyar una decisión, utilizaremos `RENTA` como ejemplo.

Aplicaremos la regla IQR para identificar valores fuera del rango esperado y, en este ejercicio,
crearemos una versión ajustada de `RENTA` llevando esos valores al límite más cercano.

De esta manera **no eliminamos clientes**, pero reducimos la influencia de los valores extremos.


In [ ]:
# Por ejemplo revisamos RENTA
outliers_renta, limite_inferior, limite_superior = detectar_outliers_iqr(df,"RENTA")

print("\nCandidatos por IQR:", len(outliers_renta))

In [ ]:
# Revisamos los valores detectados antes de tomar una decisión.
outliers_renta[["CLIENT_ID", "RENTA"]].head()

Los valores detectados no tienen por qué ser errores. Sin embargo, para este ejercicio decidimos
**limitar su influencia** en los análisis posteriores.

Crearemos `RENTA_AJUSTADA`:

- los valores dentro del rango se mantienen;
- los valores menores al límite inferior se llevan al límite inferior;
- los valores mayores al límite superior se llevan al límite superior.

La variable original `RENTA` se conserva por ahora para poder comparar ambas representaciones.


In [ ]:
# Creamos una copia de RENTA para realizar el ajuste.
df["RENTA_AJUSTADA"] = df["RENTA"].copy()

# Los valores menores al límite se llevan al límite inferior.
df.loc[
    df["RENTA_AJUSTADA"] < limite_inferior,
    "RENTA_AJUSTADA"
] = limite_inferior

# Los valores mayores al límite se llevan al límite superior.
df.loc[
    df["RENTA_AJUSTADA"] > limite_superior,
    "RENTA_AJUSTADA"
] = limite_superior

### <font color=blue>Verificamos el efecto de la decisión</font>


In [ ]:
# Comparamos gráficamente la variable original y la versión ajustada.
df[["RENTA", "RENTA_AJUSTADA"]].plot(
    kind="box",
    subplots=True,
    layout=(1, 2),
    sharey=True,
    figsize=(6, 3)
)

plt.tight_layout()
plt.show()


Una vez entendido el procedimiento con `RENTA`, aplicaremos el mismo criterio a otras variables numéricas seleccionadas.

En este ejercicio decidimos limitar la influencia de sus valores extremos utilizando los límites del IQR.


In [ ]:
def ajustar_outliers_iqr(df, columna, factor=1.5):

    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)

    iqr = q3 - q1

    limite_inferior = q1 - factor * iqr
    limite_superior = q3 + factor * iqr

    df.loc[
        df[columna] < limite_inferior,
        columna
    ] = limite_inferior

    df.loc[
        df[columna] > limite_superior,
        columna
    ] = limite_superior

In [ ]:
# Variables numéricas en las que decidimos limitar
# la influencia de valores extremos.
# RENTA no se incluye porque ya fue tratada mediante RENTA_AJUSTADA.

variables_ajustar = [
    "CAPDEUDA",
    "USO_SMS",
    "DEUDA_ANT",
    "MIN_CALLCENTER",
    "USO_MB",
    "DEUDA_ACT",
    "MIN_ENT_3M",
    "MIN_SAL_3M"
]

for columna in variables_ajustar:
    ajustar_outliers_iqr(df, columna)


In [ ]:
# Verificamos el resultado del tratamiento.
df[variables_ajustar].plot(
    kind="box",
    subplots=True,
    layout=(2, 4),
    sharex=True,
    sharey=False,
    figsize=(8,3)
)

plt.tight_layout()
plt.show()


<font color=red>**Recuerda:** IQR primero identifica valores que merecen revisión.
Usar sus límites para ajustar la variable es una decisión adicional que debe justificarse.
Otra regla posible de detección es el z-score, pero no es necesario aplicar todas las reglas a la vez.</font>


### <font color=blue>7.6 Revisamos posibles redundancias entre variables numéricas</font>

La correlación permite observar si algunas variables cuantitativas contienen información muy similar.
Una correlación alta no obliga a eliminar una variable, pero puede indicar que conviene revisar si ambas son necesarias.


In [ ]:
# Seleccionamos las variables numéricas.
df_num = df.select_dtypes(include=np.number).copy()

# Excluimos el identificador, la variable objetivo,
# la RENTA original y la variable binaria USA_APP.
df_num = df_num.drop(
    columns=["CLIENT_ID", "TARGET", "RENTA", "USA_APP"],
    errors="ignore"
)

# Calculamos la matriz de correlación.
corr_matrix = df_num.corr()

# Mostramos solo una mitad porque la matriz es simétrica.
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(9, 7))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="Blues"
)

plt.title("Correlación entre variables numéricas")
plt.show()


Revisamos si existen variables con correlaciones muy altas, ya que podrían estar aportando información redundante.

Si no observamos correlaciones suficientemente altas como para sugerir redundancia, conservamos esas variables por ahora.

<font color=red>**Recuerda:** una correlación alta es una señal para revisar redundancia;
no es una regla automática para eliminar variables.</font>


## <font color=green>8. Exploramos las variables categóricas</font>

Primero identificamos las columnas almacenadas como texto. Luego usamos el diccionario de datos
para interpretar si son nominales, ordinales o binarias.

<font color=red>**Recuerda:** `select_dtypes(include="object")` identifica columnas almacenadas como texto,
pero no necesariamente todas las variables categóricas conceptuales.</font>


In [ ]:
# Seleccionamos las variables almacenadas como tipo object.
columnas_categoricas =  df.select_dtypes(include="object").columns.tolist()

print(columnas_categoricas)

In [ ]:
# Recorre cada columna categórica de la lista
for columna in columnas_categoricas:

    # Muestra el nombre de la columna
    print(f"\n{columna}")

    # Cuenta cuántas veces aparece cada categoría,
    # incluyendo los valores faltantes (NaN)
    display(
        df[columna]
        .value_counts(dropna=False)  # Frecuencia de cada valor
        .to_frame("n")               # Convierte el resultado en DataFrame
    )

### <font color=blue>GENERO: conservar el dato desconocido</font>

En `GENERO` existen pocos faltantes. No tenemos evidencia para asignarlos a `M` o `F`,
por lo que es más transparente conservar una categoría que indique que el dato no fue registrado.


In [ ]:
df["GENERO"] = df["GENERO"].fillna("Desconocido")

### <font color=blue>ESTADOCIVIL: demasiada información ausente</font>

`ESTADOCIVIL` tiene una proporción muy alta de valores faltantes.
En este ejercicio no contamos con otra fuente que permita recuperarlos, por lo que
no utilizaremos esta variable en la matriz analítica.


In [ ]:
porcentaje_ec = df["ESTADOCIVIL"].isna().mean() * 100

print(
    "Porcentaje de faltantes:",
    round(porcentaje_ec, 2),
    "%"
)

df = df.drop(columns="ESTADOCIVIL")

### <font color=blue>DEPARTAMENTO: la conservamos por ahora</font>

`DEPARTAMENTO` es nominal y tiene varias categorías.
No la codificaremos automáticamente. Primero continuaremos con el análisis y luego decidiremos
si realmente debe formar parte de la matriz final.

<font color=red>**Recuerda:** que una variable sea nominal no significa que deba aplicarse
one-hot encoding de manera automática.</font>


## <font color=green>9. Transformaciones y construcción de la matriz analítica</font>

Con los principales problemas de calidad ya tratados, podemos construir nuevas representaciones
que ayuden al análisis.

Después de crear una variable conviene volver a explorarla: no toda transformación aporta
necesariamente información útil.


### <font color=blue>9.1 Variación de deuda</font>

En lugar de mirar solo dos montos separados, podemos preguntar:

> **¿la deuda del cliente aumentó o disminuyó respecto del mes anterior?**


In [ ]:
df["VARIACION_DEUDA"] = (df["DEUDA_ACT"]- df["DEUDA_ANT"])

df[["DEUDA_ANT", "DEUDA_ACT", "VARIACION_DEUDA"]].head()

In [ ]:
plt.figure(figsize=(6, 3))

sns.histplot(
    data=df,
    x="VARIACION_DEUDA",
    bins=25
    #color="blue"
)

plt.title("Distribución de la variación de deuda")
plt.show()

plt.figure(figsize=(5, 3))

sns.boxplot(
    data=df,
    x="TARGET",
    y="VARIACION_DEUDA"
)

plt.title("Variación de deuda según abandono")
plt.show()


La nueva variable permite observar directamente el cambio entre periodos.
Por ejemplo en el **histograma**, `VARIACION_DEUDA` está aproximadamente centrada en 0 y tiene una forma bastante simétrica. Eso significa que, en general, hay clientes cuya deuda aumenta y otros cuya deuda disminuye, pero la mayoría presenta cambios relativamente pequeños. Los valores muy positivos o muy negativos son poco frecuentes.

En el **boxplot** inferior, se separa la `VARIACION_DEUDA` según el TARGET:
* `TARGET = 0`: clientes que no abandonaron.
* `TARGET = 1`: clientes que abandonaron.
Esto permite preguntar si ¿La VARIACION_DEUDA tiene un comportamiento diferente dependiendo de si el cliente abandonó o no?
Lo que se observa es que las dos distribuciones son muy parecidas. En ambos grupos:
- la mediana está aproximadamente en 0;
- el rango intercuartílico es casi el mismo;
- los bigotes alcanzan valores similares;
- aparecen valores atípicos tanto positivos como negativos.
Por tanto, una interpretación adecuada sería:

>La variación de la deuda presenta una distribución muy similar entre los clientes que abandonaron y los que no abandonaron. **No se observa visualmente una diferencia importante entre ambos grupos, por lo que esta variable, considerada de manera aislada, parece tener poca capacidad para discriminar el abandono.** Es decir: *el cambio de deuda por sí solo no parece explicar el abandono*.

Además, aparece una decisión para la matriz final:
si conservamos `DEUDA_ACT` y `VARIACION_DEUDA`, `DEUDA_ANT` puede reconstruirse,
por lo que mantener las tres variables produciría redundancia exacta.


## <font color=green>10. Discretización: facilitar comparaciones por grupos</font>

La discretización puede ser útil cuando queremos comparar segmentos de forma más interpretable.

Como no tenemos puntos de corte de negocio para `ANTIGUEDAD`, utilizaremos `qcut()`,
que forma grupos según la posición de los datos en la distribución.


In [ ]:
df["ANTIGUEDAD_GRUPO"] = pd.qcut(
    df["ANTIGUEDAD"],
    q=3,
    labels=["Baja", "Media", "Alta"]
)

df[["ANTIGUEDAD", "ANTIGUEDAD_GRUPO"]].head()

In [ ]:
# Agrupa a los clientes según su categoría de antigüedad
resumen_antiguedad = (
    df
    .groupby(
        "ANTIGUEDAD_GRUPO",
        observed=False   # Mantiene todas las categorías definidas, incluso si alguna no aparece
    )
    .agg(
        # Cuenta cuántos clientes hay en cada grupo de antigüedad
        n_clientes=("CLIENT_ID", "size"),

        # Calcula la proporción de abandono en cada grupo.
        # Como TARGET toma valores 0 y 1, su media equivale a la tasa de abandono.
        tasa_abandono=("TARGET", "mean")
    )
    # Convierte el índice generado por groupby nuevamente en una columna
    .reset_index()
)

# Muestra la tabla resumen
display(resumen_antiguedad)


plt.figure(figsize=(5, 3))

# Grafica la tasa de abandono para cada grupo de antigüedad
sns.barplot(
    data=resumen_antiguedad,
    x="ANTIGUEDAD_GRUPO",
    y="tasa_abandono"
    #color="blue"
)

plt.ylabel("Proporción de abandono")
plt.title("Abandono por grupo de antigüedad")
plt.show()

Se observa que el grupo de `antigüedad baja` presenta una tasa de abandono de aproximadamente 41%, bastante mayor que la de los grupos de `antigüedad media` y `alta`, que están alrededor de 28% y 29%, respectivamente. Lo que sugiere que los clientes más recientes presentan una mayor propensión al abandono.

**Nota:** esta variable parece bastante más interesante que `VARIACION_DEUDA`, porque aquí sí se observa una separación visible entre categorías del predictor respecto al `TARGET`.

<font color=red>**Recuerda:** `qcut()` define los cortes según cuantiles.
`cut()` sería más apropiado si los límites estuvieran definidos por el negocio o por una regla externa.</font>

La discretización facilita la comparación de segmentos, pero pierde parte del detalle
de la variable original.

In [ ]:
# Antes de integrar esta variable necesitamos
# convertir la categoría ordinal a valores numéricos
mapa_antiguedad = {
    "Baja": 0,
    "Media": 1,
    "Alta": 2
}

df["ANTIGUEDAD_GRUPO_ORD"] = (
    df["ANTIGUEDAD_GRUPO"]
    .map(mapa_antiguedad)
)

## <font color=green>11. Codificación de variables categóricas</font>

`GENERO` es nominal. Para incorporarla a una matriz numérica utilizaremos one-hot encoding.


In [ ]:
# Aplicamos One-Hot Encoding a la variable GENERO
df_transformado = pd.get_dummies(
    df,
    columns=["GENERO"],
    prefix="genero",
    dtype=int
)

# Columnas creadas para GENERO
columnas_genero = ["genero_F", "genero_M"]

print(columnas_genero)

In [ ]:
display(df_transformado[columnas_genero].head())

### <font color=blue>¿Y DEPARTAMENTO?</font>

`DEPARTAMENTO` también es nominal, pero tiene varias categorías.
En esta versión del laboratorio no lo incorporaremos a la matriz final para mantener el ejemplo manejable.

En un proyecto real se mantiene la variable DEPARTAMENTO porque puede ser útil posteriormente para enriquecer la información geográfica de los clientes, por ejemplo mediante su vinculación con códigos UBIGEO, coordenadas o variables de localización.


In [ ]:
print("Número de departamentos:", df["DEPARTAMENTO"].nunique())

df["DEPARTAMENTO"].value_counts()

## <font color=green>12. Construimos la matriz analítica</font>

La matriz final no contiene necesariamente todas las variables disponibles ni todas las
variables creadas durante el EDA.

Para este ejercicio tomaremos las siguientes decisiones:

- `CLIENT_ID`: se conserva aparte para trazabilidad.
- `RENTA`: se reemplaza por `RENTA_AJUSTADA`, que limita la influencia de los valores extremos detectados con IQR.
- `TARGET`: se conserva aparte como variable de interés.
- `DEUDA_ANT`: se excluye porque conservamos `DEUDA_ACT` y `VARIACION_DEUDA`.
- `USO_APP`: se reemplaza por la representación binaria `USA_APP`.
- `DEPARTAMENTO`: no se incluye en esta versión.
- `ANTIGUEDAD_GRUPO`: se usó para EDA; en la matriz conservamos `ANTIGUEDAD`.


In [ ]:
def construir_matriz_analitica(df):

    # La función implementa decisiones ya tomadas;
    # no decide automáticamente qué eliminar.
    base = df.copy()

    columnas_excluir = [
        "CLIENT_ID",
        "TARGET",
        "RENTA",
        "DEUDA_ANT",
        "USO_APP",
        "DEPARTAMENTO",
        "ANTIGUEDAD_GRUPO"
    ]

    return base.drop(
        columns=columnas_excluir,
        errors="ignore"
    )


id_cliente = df_transformado["CLIENT_ID"].copy()
target = df_transformado["TARGET"].copy()

matriz_analitica = construir_matriz_analitica(
    df_transformado
)

print("Dimensiones de la matriz:", matriz_analitica.shape)
display(matriz_analitica.head())


<font color=red>**Recuerda:** crear una variable durante el EDA no significa que deba entrar
automáticamente en la matriz final.</font>


## <font color=green>13. Revisamos si necesitamos escalamiento</font>

Antes de escalar, observamos los rangos de las variables cuantitativas.

No escalaremos **identificadores**, variables **binarias** ni columnas **one-hot**.


In [ ]:
variables_escalar = [
    "RENTA_AJUSTADA",
    "USO_SMS",
    "MIN_CALLCENTER",
    "ANTIGUEDAD",
    "USO_MB",
    "DEUDA_ACT",
    "MIN_ENT_3M",
    "MIN_SAL_3M",
    "CAPDEUDA",
    "EDAD",
    "VARIACION_DEUDA"
]

display(
    matriz_analitica[
        variables_escalar
    ].describe().T[
        ["min", "max", "mean", "std"]
    ]
)


### <font color=blue>Comparamos Min–Max y z-score</font>

Creamos dos representaciones para observar cómo cambia la escala.
Esto no significa que debamos utilizar ambas.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

minmax = MinMaxScaler()
estandarizador = StandardScaler()

matriz_minmax = matriz_analitica.copy()
matriz_zscore = matriz_analitica.copy()

matriz_minmax[variables_escalar] = (
    minmax.fit_transform(
        matriz_minmax[variables_escalar]
    )
)

matriz_zscore[variables_escalar] = (
    estandarizador.fit_transform(
        matriz_zscore[variables_escalar]
    )
)

print("Sin escalar:")
display(matriz_analitica.head(3))

print("Min–Max:")
display(matriz_minmax.head(3))

print("z-score:")
display(matriz_zscore.head(3))


- **Min–Max:** lleva normalmente cada variable al rango `[0,1]`.
- **z-score:** expresa cada valor en desviaciones estándar respecto de la media.
- **La elección depende de la técnica analítica posterior**.

<font color=red>**Recuerda:** escalar cambia la representación; no corrige faltantes ni outliers.</font>


## <font color=green>14. Resultado del preprocesamiento</font>

In [ ]:
X = matriz_minmax   # Si se decide usar escalamiento Min-Max
# X = matriz_zscore # Alternativa: estandarización

y = target

print("Información de la matriz final:")
X.info()

print("\nMatriz final para modelamiento:")
display(X.head())

print("Dimensiones:", X.shape)

## <font color=green>14. ¿Qué preguntas analíticas podemos plantear?</font>

A partir de este caso podrían surgir preguntas como:

1. ¿Los clientes que abandonan presentan una renta diferente?
2. ¿La variación de deuda muestra comportamientos distintos entre quienes abandonan y quienes no?
3. ¿Existe diferencia en abandono entre clientes que usan y no usan la aplicación?
4. ¿La intensidad de uso aporta información adicional frente a `USA_APP`?
5. ¿Los grupos de antigüedad presentan proporciones de abandono diferentes?
6. ¿Los valores extremos de renta corresponden a errores o a clientes válidos poco frecuentes?
7. ¿Qué variables originales dejan de ser necesarias después de crear nuevas características?

Estas preguntas hacen que el EDA y la transformación tengan un propósito porque **no graficamos ni transformamos por rutina sino que lo hacemos para entender mejor el problema.**


## <font color=green>15. Aplicación al proyecto</font>

Después del laboratorio, aplique el mismo razonamiento a su proyecto.

### <font color=blue>A. Contexto y variables</font>
Revise el significado y tipo conceptual de las variables que utilizará.

### <font color=blue>B. Calidad</font>
Seleccione al menos dos problemas de calidad y explique qué decisión tomó y por qué.

### <font color=blue>C. EDA</font>
Plantee al menos **tres preguntas analíticas** y genere visualizaciones que ayuden a investigarlas.
Cada gráfico debe tener una interpretación breve.

### <font color=blue>D. Outliers</font>
Aplique IQR (u otra regla vista en clase) a una variable numérica relevante. Identifique los candidatos y tome una decisión justificada: conservar, ajustar, corregir o excluir.

### <font color=blue>E. Variables derivadas</font>
Construya al menos una variable derivada y compruebe mediante EDA si aporta una lectura útil.

### <font color=blue>F. Discretización</font>
Evalúe si alguna variable podría ser útil en grupos.
Si discretiza, justifique si corresponde `cut()` o `qcut()`.

### <font color=blue>G. Matriz analítica</font>
Indique qué variables conservaría, transformaría, reemplazaría, excluiría o escalaría.

<font color=red>**Recuerda:** no es obligatorio aplicar todas las técnicas.
Se evalúa que las decisiones sean coherentes con los datos, la unidad de análisis y el objetivo del proyecto.</font>
